In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
import matplotlib.animation as animation
from ipywidgets import interact, FloatSlider, widgets
from IPython.display import display

# ==========================================
# SIMULATION CONFIGURATION (Global Variables)
# ==========================================
sa = 30           # Steering angle limit in degrees
wb = 2            # Wheelbase in meters
k_gain = 0.5      # Default position gain
ks_gain = 1.0     # Default softening gain
v_target = 10.0   # Target speed (m/s)
lat_offset = 20.0 # Initial lateral offset from path (meters)
time_limit = 20.0 # Simulation run time (seconds)
dt = 0.05         # Physics time step (seconds)
gains_to_test = [0.1, 0.5, 1.5, 2.0]  # Different k gains for experimentation

# ==========================================
# VEHICLE & CONTROLLER CLASSES
# ==========================================
class KinematicBicycleModel:
    def __init__(self, x=0.0, y=0.0, yaw=0.0, v=0.0, L=2.0): 
        self.x = x
        self.y = y
        self.yaw = yaw
        self.v = v
        self.L = L  

    def update(self, a, delta, dt, max_steer_deg): 
        delta = np.clip(delta, -np.radians(max_steer_deg), np.radians(max_steer_deg))
        
        self.x += self.v * np.cos(self.yaw) * dt
        self.y += self.v * np.sin(self.yaw) * dt
        self.yaw += (self.v / self.L) * np.tan(delta) * dt
        self.v += a * dt

class StanleyController:
    def __init__(self, k=0.5, ks=1.0): 
        self.k = k
        self.ks = ks

    def compute_steering(self, vehicle, path_x, path_y, path_yaw):
        # 1. Find the closest point on the path to the FRONT axle
        fx = vehicle.x + vehicle.L * np.cos(vehicle.yaw)
        fy = vehicle.y + vehicle.L * np.sin(vehicle.yaw)
        
        dx = [fx - x for x in path_x]
        dy = [fy - y for y in path_y]
        distances = np.hypot(dx, dy)
        target_idx = np.argmin(distances)

        # 2. Correct Cross-Track Error calculation (Using 2D Cross Product)
        dx_front = fx - path_x[target_idx]
        dy_front = fy - path_y[target_idx]
        
        path_vec_x = np.cos(path_yaw[target_idx])
        path_vec_y = np.sin(path_yaw[target_idx])
        
        error_front_axle = dx_front * path_vec_y - dy_front * path_vec_x

        # 3. Correct Heading Error (Path Yaw minus Vehicle Yaw)
        yaw_error = path_yaw[target_idx] - vehicle.yaw
        yaw_error = np.arctan2(np.sin(yaw_error), np.cos(yaw_error))

        # 4. Stanley Control Law
        crosstrack_steering = np.arctan2(self.k * error_front_axle, vehicle.v + self.ks)
        delta = yaw_error + crosstrack_steering
        
        return delta, error_front_axle

# ==========================================
# SIMULATION CORE ENGINE
# ==========================================
def run_simulation(path_x, path_y, path_yaw, k_param, ks_param):
    vehicle = KinematicBicycleModel(x=path_x[0], y=path_y[0] - lat_offset, yaw=path_yaw[0], v=2.0, L=wb)
    controller = StanleyController(k=k_param, ks=ks_param)
    
    time = 0.0
    logs = []

    while time < time_limit:
        # Calculate lateral steering command first
        delta, cte = controller.compute_steering(vehicle, path_x, path_y, path_yaw)
        
        # REALISTIC TWEAK: Reduce target speed based on how hard the car is steering
        # If steering is at 0, target is v_target. If steering is maxed out, target speed drops.
        dynamic_v_target = v_target * np.cos(delta) 
        
        # Alternatively, we can add a 'tire scrub' penalty directly to the acceleration:
        # Tire scrub power loss increases drastically with steering angle
        tire_scrub_penalty = 1.2 * np.sin(abs(delta)) * vehicle.v
        
        # Simple P-controller adjusted with our new physical limitations
        a = 0.5 * (dynamic_v_target - vehicle.v) - tire_scrub_penalty
        
        # Append state data to logs
        logs.append({
            'time': time, 'x': vehicle.x, 'y': vehicle.y, 
            'yaw': vehicle.yaw, 'v': vehicle.v, 'cte': cte, 'delta': delta
        })
        
        # Step physics forward
        vehicle.update(a, delta, dt, max_steer_deg=sa)
        time += dt
        
    return pd.DataFrame(logs)

# ==========================================
# EXECUTION & EXPERIMENTATION
# ==========================================

# 1. Define raw track waypoints and build cubic spline
waypoints_x = [0.0, 20.0,  40.0, 60.0,  80.0, 100.0, 110.0, 100.0, 80.0]
waypoints_y = [0.0,  5.0, -10.0,  0.0,  15.0,  10.0,  -5.0, -20.0, -25.0]

t_waypoints = np.arange(len(waypoints_x))
spline_x = CubicSpline(t_waypoints, waypoints_x, bc_type='natural')
spline_y = CubicSpline(t_waypoints, waypoints_y, bc_type='natural')

t_fine = np.linspace(0, len(waypoints_x) - 1, 1000)
path_x = spline_x(t_fine)
path_y = spline_y(t_fine)

dx_dt = spline_x(t_fine, 1)
dy_dt = spline_y(t_fine, 1)
path_yaw = np.arctan2(dy_dt, dx_dt)

# 2. Execute batch runs
results = {}
for g in gains_to_test:
    run_id = f"Gain_k_{g}"
    results[run_id] = run_simulation(path_x, path_y, path_yaw, k_param=g, ks_param=ks_gain)

# Print out static diagnostic data once
for run_id, df in results.items():
    rmse = np.sqrt((df['cte'] ** 2).mean())
    print(f"{run_id} -> Average Cross-Track Error: {df['cte'].abs().mean():.3f}m, RMSE: {rmse:.3f}m")

# 3. Create the Interactive Plotting Function
def update_dashboard(target_time):
    # Setup the 2x2 grid figure
    fig, axs = plt.subplots(2, 2, figsize=(15, 11))
    
    # --- Subplot 1: Visual Map Tracker (Top Left - Row 0, Col 0) ---
    axs[0, 0].plot(path_x, path_y, 'r--', linewidth=2, label='Smoothed Reference Path')
    axs[0, 0].plot(waypoints_x, waypoints_y, 'ko', markersize=6, label='Raw Waypoints')
    
    for run_id, df in results.items():
        line, = axs[0, 0].plot(df['x'], df['y'], label=f"{run_id}", alpha=0.6)
        closest_idx = (df['time'] - target_time).abs().argmin()
        current_row = df.iloc[closest_idx]
        axs[0, 0].plot(current_row['x'], current_row['y'], marker='o', markersize=10, 
                       color=line.get_color(), markeredgecolor='black', markeredgewidth=1.5)
        
    axs[0, 0].set_title(f'Vehicle Positions at t = {target_time:.2f}s')
    axs[0, 0].set_xlabel('X Position (m)')
    axs[0, 0].set_ylabel('Y Position (m)')
    axs[0, 0].legend()
    axs[0, 0].grid(True)
    axs[0, 0].axis('equal') 

    # --- Subplot 2: Cross-Track Error Timeline (Top Right - Row 0, Col 1) ---
    for run_id, df in results.items():
        axs[0, 1].plot(df['time'], df['cte'], label=f"{run_id}")
    axs[0, 1].axvline(x=target_time, color='k', linestyle='--', linewidth=1.5)
    axs[0, 1].set_xlabel('Time (s)')
    axs[0, 1].set_ylabel('Cross-Track Error (m)')
    axs[0, 1].set_title('Cross-Track Error Timeline')
    axs[0, 1].grid(True)

    # --- Subplot 3: Steering Angle Timeline (Bottom Left - Row 1, Col 0) ---
    for run_id, df in results.items():
        steering_deg = np.degrees(df['delta'])
        axs[1, 0].plot(df['time'], steering_deg, label=f"{run_id}") # Fixed index to [1, 0]
    axs[1, 0].axhline(y=sa, color='grey', linestyle=':', label='Max Mech Limit')
    axs[1, 0].axhline(y=-sa, color='grey', linestyle=':')
    axs[1, 0].axvline(x=target_time, color='k', linestyle='--', linewidth=1.5)
    axs[1, 0].set_xlabel('Time (s)')
    axs[1, 0].set_ylabel('Steering Angle (Degrees)')
    axs[1, 0].set_title('Steering Input Timeline')
    axs[1, 0].grid(True)

    # --- Subplot 4: Velocity Tracking Timeline (Bottom Right - Row 1, Col 1) ---
    axs[1, 1].axhline(y=v_target, color='r', linestyle='--', linewidth=2, label='Target Speed')
    for run_id, df in results.items():
        axs[1, 1].plot(df['time'], df['v'], label=f"{run_id}")
    axs[1, 1].axvline(x=target_time, color='k', linestyle='--', linewidth=1.5)
    axs[1, 1].set_xlabel('Time (s)')
    axs[1, 1].set_ylabel('Velocity (m/s)')
    axs[1, 1].set_title('Vehicle Speed Timeline')
    axs[1, 1].grid(True)

    plt.tight_layout()
    plt.show()

# 4. Initialize and display the interactive controller widget
interact(
    update_dashboard, 
    target_time=FloatSlider(
        value=0.0, 
        min=0.0, 
        max=time_limit, 
        step=dt, 
        description='Sim Time (s):',
        continuous_update=True, # Set to False if your computer lags while sliding
        layout=widgets.Layout(width='60%')
    )
);

Gain_k_0.1 -> Average Cross-Track Error: 7.881m, RMSE: 8.926m
Gain_k_0.5 -> Average Cross-Track Error: 3.315m, RMSE: 6.067m
Gain_k_1.5 -> Average Cross-Track Error: 3.030m, RMSE: 6.303m
Gain_k_2.0 -> Average Cross-Track Error: 3.049m, RMSE: 6.381m


interactive(children=(FloatSlider(value=0.0, description='Sim Time (s):', layout=Layout(width='60%'), max=20.0…